In [ ]:
torch_device = "cuda:2"

In [ ]:
# m is the LDM metadata
# e is the embedding
# r is the random bit
import json
import hmac
import hashlib
import pickle

m = {"model_id" : "stabilityai/stable-diffusion-2-1", "timesteps": 10}
def construct_POA(m, e, r, author_id = "1"):
    m = pickle.dumps(m)
    e = pickle.dumps(e)
    
    signature = hmac.new(
        bytes("1", 'latin-1'), 
        msg=pickle.dumps((e, m, r)), 
        digestmod=hashlib.sha256
    ).hexdigest().upper()

    return signature [:8]

In [ ]:
from datasets import load_dataset
train = [sample['Prompt'] for sample in load_dataset('Gustavosta/Stable-Diffusion-Prompts')['train']]

In [ ]:
from diffusers import DDPMPipeline
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler, DDIMScheduler
from PIL import Image
import torch
from tqdm.auto import tqdm

model_id = "stabilityai/stable-diffusion-2-1"
# batch_size=10

vae = AutoencoderKL.from_pretrained(model_id , subfolder="vae", use_safetensors=True)
tokenizer = CLIPTokenizer.from_pretrained(model_id , subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    model_id , subfolder="text_encoder", use_safetensors=True
)
unet = UNet2DConditionModel.from_pretrained(
    model_id , subfolder="unet", use_safetensors=True
)

vae.to(torch_device)
text_encoder.to(torch_device)
unet.to(torch_device)

prompt= ['carrots with coffee']
batch_size=len(prompt)

text_input = tokenizer(
    prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
)

with torch.no_grad():
    text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]


max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt")
uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler", batch_size=batch_size)


In [ ]:
def convert_sentence_to_embed(lst_of_sentences):
    text_input = tokenizer(
        lst_of_sentences, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt"
    )
    batch_size = len(lst_of_sentences)
    with torch.no_grad():
        text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]
    
    
    max_length = text_input.input_ids.shape[-1]
    uncond_input = tokenizer([""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt")
    uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])
    return text_embeddings

import numpy as np
scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler", batch_size=batch_size)
    
def get_vae_embed_from_text_embed(text_embed, seed=1, timesteps=10, 
                                  n_samples=1, batch_size=batch_size,
                                 guidance_scale = 7.5,
                                 same_seed=False,
                                 latents=None): # use same seed across all iteration 
    
    
    out = []
    scheduler.set_timesteps(timesteps)
    out_latent = []
     #seed is a 64 bit signed int
    generator = torch.manual_seed(seed)
    for _ in tqdm([i for i in range(n_samples)]):
        if same_seed:
            generator = torch.manual_seed(seed)
        if latents is None:
            latents = torch.randn(
                (batch_size, unet.config.in_channels, 512//8, 512//8),
                    generator=generator,
                )
        latents = latents.to(torch_device)
        latents = latents * scheduler.init_noise_sigma
        
        for t in tqdm(scheduler.timesteps):
            
            # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
            latent_model_input = torch.cat([latents] * 2)
            latent_model_input = scheduler.scale_model_input(latent_model_input, timestep=t)

            # predict the noise residual
            with torch.no_grad():
                noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embed).sample

            # perform guidance
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

            # compute the previous noisy sample x_t -> x_t-1
            latents = scheduler.step(noise_pred, t, latents).prev_sample
        # vae step
        latents = 1 / 0.18215 * latents
        out_latent.append(latents)
    return torch.cat(out_latent,dim=0).to(torch_device)

def latent_to_img(latents):
    with torch.no_grad():
        image = vae.decode(latents).sample
    np_image = np.array(image.cpu())
#         if height % 8 != 0 or width % 8 != 0:

        # np_image = cv2.resize(np.transpose(np_image, (0,2, 3, 1)), dsize=dim_wh, interpolation=cv2.INTER_LINEAR)
    # np_image = np.transpose(np_image, (0,3,1,2))
    return np_image
    #     out.extend(np_image)
    # return np.array(out)

In [ ]:
# m is the LDM metadata
# e is the embedding
# r is the random bit
import json
import hmac
import hashlib
import pickle

m = {"model_id" : "stabilityai/stable-diffusion-2-1", "timesteps": 10}
def construct_POA(m, e, r, author_id = "1"):
    m = pickle.dumps(m)
    e = pickle.dumps(e)
    
    signature = hmac.new(
        bytes("1", 'latin-1'), 
        msg=pickle.dumps((e, m, r)), 
        digestmod=hashlib.sha256
    ).hexdigest().upper()

    return signature [:8]

In [ ]:
# ks_test_stat = []
from scipy.stats import *
params = []
n_prompts = 200
n_image = 200
base_latent = []
def compute_pairwise_dist(arr):
    return [float(torch.sum(arr[i]*arr[j])) for i in range(len(arr)) for j in range(i+1, len(arr))]
    
for nn in range(n_prompts):
    # generate images
    print(nn)
    embeds_same_prompt_diff_seed = []
    prompt_embed = convert_sentence_to_embed(train[nn:nn+1])
    for i in range(n_image):
        if i == 0:
            seed = construct_POA(m=m, e=prompt_embed, r=1)
            seed = int(seed, base=16)
        else:
            seed = i
        latents = torch.randn(
                    (batch_size, unet.config.in_channels, 512//8, 512//8),
                        generator = torch.manual_seed(seed),
                    )
        # latents = torch.randn(
        #             (batch_size, unet.config.in_channels, 512//8, 512//8),
        #                 generator = torch.manual_seed(i),
        #             )
        # lats_same_prompt_diff_seed.append(latents)
        vae_embed = get_vae_embed_from_text_embed(prompt_embed, latents=latents)
        
        embeds_same_prompt_diff_seed.append(vae_embed)
    # compute ks
    same_prompt_pairwise_dist = compute_pairwise_dist(embeds_same_prompt_diff_seed)
    ratio_1 = [same_prompt_pairwise_dist[i]/4/64/64 for i in range(len(same_prompt_pairwise_dist))]
    # approx distr
    beta,loc,scale = gennorm.fit(ratio_1)

        # get the params of the test distribution
        # get the base POA image embed
    params.append((beta, loc, scale))
    base_latent.append(embeds_same_prompt_diff_seed[0])
    # # get image and invert back to latent
    # im = latent_to_img(embeds_same_prompt_diff_seed[0])
    # inv_vae_latent = vae.to(torch_device).encode(torch.from_numpy(im.astype(np.float32)).to(torch_device))

    # T = torch.sum(embeds_same_prompt_diff_seed[0] * inv_vae_latent.latent_dist.mean)
    # p_val.append(gennorm.sf(float(T), beta=beta, loc=loc, scale=scale))
    # print(p_val[-1])

In [ ]:
import io
def jpeg_compress(image_arr, jpeg_quality=75):
    im_pil = Image.fromarray(image_arr.astype(np.uint8))
    buffered = io.BytesIO()
    im_pil.save(buffered, format="JPEG", quality=jpeg_quality, optimize=True)
    # compressed_image_data = buffered.getvalue()
    return np.array(Image.open(buffered))
torch.manual_seed(1)
np.random.seed(1)
sd_ori_imgs = [latent_to_img(i) for i in base_latent]
gauss_unit_noise = np.random.normal(size=sd_ori_imgs[-1].shape)
def normalize_image(im):
    return (im - np.min(im)) / (np.max(im) - np.min(im)) * 255
    # return im/255

def reset_image(im, old_min, old_max):
    # return im*255
    return im/255 * (old_max - old_min) + old_min

vae_rep = []
vae_rep_gauss_1 = []
vae_rep_gauss_3 = []
vae_rep_gauss_5 = []
vae_rep_gauss_7 = []
vae_rep_gauss_9 = []
vae_rep_jpeg_75 = []
vae_rep_jpeg_50 = []

In [ ]:

for i in sd_ori_imgs:
    with torch.no_grad():
        vae_rep.append(vae.to(torch_device).encode(torch.from_numpy(i).to(torch_device)))

        old_max, old_min = np.max(i), np.min(i)
        im_normalized = normalize_image(i)
        im_gauss_1 = im_normalized + gauss_unit_noise
        im_gauss_1 = reset_image(im_gauss_1, old_min, old_max)

        im_gauss_3 = im_normalized + gauss_unit_noise * 3**0.5
        im_gauss_3 = reset_image(im_gauss_3, old_min, old_max)

        im_gauss_5 = im_normalized + gauss_unit_noise * 5**0.5
        im_gauss_5 = reset_image(im_gauss_5, old_min, old_max)

        im_gauss_7 = im_normalized + gauss_unit_noise * 7**0.5
        im_gauss_7 = reset_image(im_gauss_7, old_min, old_max)

        im_gauss_9 = im_normalized + gauss_unit_noise * 9**0.5
        im_gauss_9 = reset_image(im_gauss_9, old_min, old_max)

        vae_rep_gauss_1.append(vae.to(torch_device).encode(torch.from_numpy(im_gauss_1.astype(np.float32)).to(torch_device)))
        vae_rep_gauss_3.append(vae.to(torch_device).encode(torch.from_numpy(im_gauss_3.astype(np.float32)).to(torch_device)))
        vae_rep_gauss_5.append(vae.to(torch_device).encode(torch.from_numpy(im_gauss_5.astype(np.float32)).to(torch_device)))
        vae_rep_gauss_7.append(vae.to(torch_device).encode(torch.from_numpy(im_gauss_7.astype(np.float32)).to(torch_device)))
        vae_rep_gauss_9.append(vae.to(torch_device).encode(torch.from_numpy(im_gauss_9.astype(np.float32)).to(torch_device)))

        # jpeg compression
        im_jpeg_75 = jpeg_compress(np.transpose(im_normalized[0],(1,2,0)), jpeg_quality=75)
        im_jpeg_75 = np.transpose(im_jpeg_75, (2, 0, 1))
        im_jpeg_75 = reset_image(im_jpeg_75, old_min, old_max)
        im_jpeg_75 = im_jpeg_75.reshape(im_normalized.shape)
        vae_rep_jpeg_75.append(vae.to(torch_device).encode(torch.from_numpy(im_jpeg_75.astype(np.float32)).to(torch_device)))

        im_jpeg_50 = jpeg_compress(np.transpose(im_normalized[0],(1,2,0)), jpeg_quality=20)
        im_jpeg_50 = np.transpose(im_jpeg_50, (2, 0, 1))
        im_jpeg_50 = reset_image(im_jpeg_50, old_min, old_max)
        im_jpeg_50 = im_jpeg_50.reshape(im_normalized.shape)
        vae_rep_jpeg_50.append(vae.to(torch_device).encode(torch.from_numpy(im_jpeg_50.astype(np.float32)).to(torch_device)))

In [ ]:
def get_normlized_latent(lat):
    return lat/torch.std(lat)

T = lambda x, y: float(torch.sum(x*y)/4/64/64)

test_stat = []
test_stat_gauss_1, test_stat_gauss_3,test_stat_gauss_5,test_stat_gauss_7, test_stat_gauss_9, test_stat_jpeg_50, test_stat_jpeg_75  = [], [], [], [], [], [], []
test_stat_wrong = []
p_gauss_1, p_gauss_3,p_gauss_5,p_gauss_7, p_gauss_9, p_jpeg_50, p_jpeg_75  = [], [], [], [], [], [], []
p_clean = []
for i in range(len(vae_rep)):
    # get distr params
    beta, loc, scale = params[i]
    
    # normalize both
    latent_1 = vae_rep[i].latent_dist.mean
    latent_2 = base_latent[i]
    # latent_1 = get_normlized_latent(latent_1)
    # latent_2 = get_normlized_latent(latent_2)
    test_stat.append(T(latent_1, latent_2))
    p_clean.append(gennorm.sf(float(test_stat[-1]), beta=beta, loc=loc, scale=scale))
    
    latent_wrong = base_latent[i-2]
    latent_wrong = latent_wrong/torch.std(latent_wrong)
    test_stat_wrong.append(float(torch.sum((latent_2 * latent_wrong))))

    
    
    # noisy latent 
    
    latent_gauss_1 = vae_rep_gauss_1[i].latent_dist.mean
    # latent_gauss_1 = get_normlized_latent(latent_gauss_1 )
    test_stat_gauss_1.append(T(latent_gauss_1, latent_2))
    p_gauss_1.append(gennorm.sf(float(test_stat_gauss_1[-1]), beta=beta, loc=loc, scale=scale))
    
    latent_gauss_3 = vae_rep_gauss_3[i].latent_dist.mean
    # latent_gauss_3 = get_normlized_latent(latent_gauss_3 )
    test_stat_gauss_3.append(T(latent_gauss_3, latent_2))
    p_gauss_3.append(gennorm.sf(float(test_stat_gauss_3[-1]), beta=beta, loc=loc, scale=scale))

    latent_gauss_5 = vae_rep_gauss_5[i].latent_dist.mean
    # latent_gauss_5 = get_normlized_latent(latent_gauss_5)
    test_stat_gauss_5.append(T(latent_gauss_5, latent_2))
    p_gauss_5.append(gennorm.sf(float(test_stat_gauss_5[-1]), beta=beta, loc=loc, scale=scale))
    
    latent_gauss_7 = vae_rep_gauss_7[i].latent_dist.mean
    # latent_gauss_7 = get_normlized_latent(latent_gauss_7 )
    test_stat_gauss_7.append(T(latent_gauss_7, latent_2))
    p_gauss_7.append(gennorm.sf(float(test_stat_gauss_7[-1]), beta=beta, loc=loc, scale=scale))

    latent_gauss_9 = vae_rep_gauss_9[i].latent_dist.mean
    # latent_gauss_9 = get_normlized_latent(latent_gauss_9 )
    test_stat_gauss_9.append(T(latent_gauss_9, latent_2))
    p_gauss_9.append(gennorm.sf(float(test_stat_gauss_9[-1]), beta=beta, loc=loc, scale=scale))

    latent_gauss_jpeg_50 = vae_rep_jpeg_50[i].latent_dist.mean
    # latent_gauss_jpeg_50 = get_normlized_latent(latent_gauss_jpeg_50 )
    test_stat_jpeg_50.append(T(latent_gauss_jpeg_50, latent_2))
    p_jpeg_50.append(gennorm.sf(float(test_stat_jpeg_50[-1]), beta=beta, loc=loc, scale=scale))

    latent_gauss_jpeg_75 = vae_rep_jpeg_75[i].latent_dist.mean
    # latent_gauss_jpeg_75 = get_normlized_latent(latent_gauss_jpeg_75 )
    test_stat_jpeg_75.append(T(latent_gauss_jpeg_75, latent_2))
    p_jpeg_75.append(gennorm.sf(float(test_stat_jpeg_75[-1]), beta=beta, loc=loc, scale=scale))

In [ ]:
latent_gauss_1

In [ ]:
latent_2

In [ ]:
2**-50

In [ ]:
lg2 = np.log2(p_clean)
print("clean")
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))

print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
lg2 = np.log2(p_gauss_1)
print("Gauss N(0, 1)")
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))
print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
lg2 = np.log2(p_gauss_3)
print("Gauss N(0, 3)")
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))

print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
print("Gauss N(0, 5)")
lg2 = np.log2(p_gauss_5)
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))
print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
print("Gauss N(0, 7)")
lg2 = np.log2(p_gauss_7)
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))

print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
print("Gauss N(0, 9)")
lg2 = np.log2(p_gauss_9)
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))
print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
print("JPEG Quality 75")
lg2 = np.log2(p_jpeg_75)
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))
print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
print("JPEG Quality 50")
lg2 = np.log2(p_jpeg_50)
print("mean, std: ", np.mean(lg2), np.std(lg2))
print("min, max: ", np.min(lg2), np.max(lg2))

print("False reject Rate")
print("alpha = 2^-30", np.mean([i > -30 for i in lg2]))
print("alpha = 2^-40", np.mean([i > -40 for i in lg2]))
print("alpha = 2^-50", np.mean([i > -50 for i in lg2]))

In [ ]:
2**-25

In [ ]:
import dill
dill.dump_session('3_poa_perturb_gauss_jpeg.db')
# dill.load_session('notebook_env.db')
